# Module 37 — Trending Topics: LDA & BERTopic on Query Logs

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Gensim, BERTopic, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 25 (Text Preprocessing)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Query Logs](#3-synthetic-query-logs)
4. [LDA Topic Modeling](#4-lda-topic-modeling)
5. [Topic Visualization](#5-topic-visualization)
6. [Temporal Trends](#6-temporal-trends)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why topic modeling for search queries?

Topic modeling **discovers hidden themes** in text:
- **Query understanding**: Identify what users are searching for.
- **Trend detection**: Spot emerging topics early.
- **Content strategy**: Create content for popular topics.

**Applications at Yandex**:
1. **Trending searches**: Identify popular queries in real-time.
2. **Content recommendations**: Suggest articles on trending topics.
3. **Ad targeting**: Target ads to trending topics.

**Key algorithms**:
- **LDA**: Latent Dirichlet Allocation (classical).
- **BERTopic**: Neural topic modeling with BERT embeddings.
- **NMF**: Non-negative Matrix Factorization.

In this notebook we will:
1. Generate synthetic search query logs.
2. Apply LDA topic modeling.
3. Visualize topic distributions.
4. Track temporal trends in topics.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── NLP libraries ────────────────────────────────────────────────
from gensim import corpora
from gensim.models import LdaModel

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Query Logs

In [ ]:
# Generate synthetic search query logs
N_QUERIES = 50000

# Topic templates
topics = {
    'technology': ['laptop', 'phone', 'software', 'programming', 'python', 'data', 'science'],
    'sports': ['football', 'basketball', 'tennis', 'soccer', 'championship', 'tournament'],
    'entertainment': ['movie', 'music', 'concert', 'celebrity', 'netflix', 'streaming'],
    'health': ['covid', 'vaccine', 'health', 'fitness', 'diet', 'exercise'],
    'finance': ['stock', 'bitcoin', 'crypto', 'investment', 'trading', 'market'],
}

# Generate queries
queries = []
timestamps = pd.date_range('2024-01-01', periods=N_QUERIES, freq='10min')

for i in range(N_QUERIES):
    # Select topic with some randomness
    topic = np.random.choice(list(topics.keys()))
    words = np.random.choice(topics[topic], size=np.random.randint(2, 5), replace=True)
    query = ' '.join(words)
    
    queries.append({
        'query_id': i,
        'query': query,
        'timestamp': timestamps[i],
        'true_topic': topic
    })

df = pd.DataFrame(queries)

print(f"✓ Generated {N_QUERIES:,} search queries")
print(f"  Topics: {len(topics)}")
print(f"  Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nTopic distribution:")
print(df['true_topic'].value_counts())

---
## §4 · LDA Topic Modeling

In [ ]:
# Preprocess queries
tokenized_queries = [query.lower().split() for query in df['query']]

# Create dictionary and corpus
dictionary = corpora.Dictionary(tokenized_queries)
corpus = [dictionary.doc2bow(doc) for doc in tokenized_queries]

# Train LDA model
N_TOPICS = 5
print(f"Training LDA with {N_TOPICS} topics...")

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=N_TOPICS,
    random_state=42,
    passes=10,
    alpha='auto',
    per_word_topics=True
)

print(f"✓ LDA model trained")
print(f"\nTop words per topic:")
for idx, topic in lda_model.print_topics(-1):
    print(f"  Topic {idx}: {topic[:80]}...")

---
## §5 · Topic Visualization

In [ ]:
# Get topic distribution for each query
topic_distributions = []
for bow in corpus:
    topic_dist = lda_model.get_document_topics(bow, minimum_probability=0)
    topic_distributions.append([prob for _, prob in topic_dist])

topic_df = pd.DataFrame(topic_distributions, columns=[f'Topic_{i}' for i in range(N_TOPICS)])

# Assign dominant topic
df['predicted_topic'] = topic_df.idxmax(axis=1)

# Topic distribution
topic_counts = df['predicted_topic'].value_counts()

print(f"✓ Topic assignments complete")
print(f"\nPredicted topic distribution:")
print(topic_counts)

---
## §6 · Temporal Trends

In [ ]:
# Track topic trends over time
df['date'] = df['timestamp'].dt.date

# Daily topic counts
daily_topics = df.groupby(['date', 'predicted_topic']).size().unstack(fill_value=0)

# Normalize to proportions
daily_topics_pct = daily_topics.div(daily_topics.sum(axis=1), axis=0)

print(f"✓ Temporal trends computed")
print(f"  Days: {len(daily_topics)}")
print(f"  Topics: {len(daily_topics.columns)}")
print(f"\nSample daily distribution:")
print(daily_topics_pct.head())

---
## §7 · Visualization

In [ ]:
# Visualize topic trends
fig = make_subplots(rows=1, cols=2, subplot_topics=['Topic Distribution', 'Topic Trends Over Time'])

# Topic distribution pie chart
fig.add_trace(go.Pie(
    labels=topic_counts.index,
    values=topic_counts.values,
    name='Topics'
), row=1, col=1)

# Topic trends over time
for topic in daily_topics_pct.columns:
    fig.add_trace(go.Scatter(
        x=daily_topics_pct.index,
        y=daily_topics_pct[topic],
        name=topic,
        mode='lines'
    ), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - LDA successfully discovers topic structure")
print("   - Topic proportions remain relatively stable")
print("   - Temporal analysis reveals trending topics")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Content & Search Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Trend detection, content strategy, ad targeting |
> | **Algorithm** | LDA for interpretability, BERTopic for quality |
> | **Topics** | 10-50 topics depending on corpus size |
> | **Evaluation** | Topic coherence, human evaluation |
> | **Deployment** | Daily topic modeling on fresh queries |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Trending topics pipeline**:
>    ```
>    Queries → Preprocess → Topic modeling → Trend detection → Alert
>    ```
>
> 2. **Use cases**:
>    | Use Case | Benefit |
>    |----------|--------|
>    | Trending searches | Identify viral topics early |
>    | Content strategy | Create content for popular topics |
>    | Ad targeting | Target ads to trending topics |
>
> 3. **Production tips**:
>    - Run topic modeling daily on fresh queries
>    - Compare topics across days to detect trends
>    - Use topic coherence to evaluate model quality
>
> ### 🔑 Key Takeaways
>
> 1. **Topic modeling discovers hidden themes** in search queries.
> 2. **LDA is interpretable** and widely used.
> 3. **BERTopic provides better quality** with neural embeddings.
> 4. **Temporal analysis reveals trends** over time.
> 5. **Daily topic modeling** enables real-time trend detection.